# cluefin-ta 기술적 분석 퀵스타트

cluefin-cli 의 `ta` 명령이 제공하던 분석 흐름(지표 계산 → 차트 → 해석 → ML 피처)을 노트북으로 옮긴 예제입니다.

## 사전 준비

- 리포지토리 루트의 `.env.test` 에 `KIS_APP_KEY` / `KIS_SECRET_KEY` 가 있어야 합니다.
- 실행: 리포지토리 루트에서 `uv run --with jupyter jupyter lab` 후 이 노트북을 엽니다.

## ⚠️ 주의

- `.env.test` 의 KIS 는 `KIS_ENV=prod`(실서버) 입니다 — 이 노트북은 계좌와 무관한 **시세 조회(read-only)** 만 수행하지만, 토큰 발급 자체는 실계좌 API 사용입니다. 토큰은 파일 캐시를 재사용하므로 반복 실행해도 재발급되지 않습니다.
- KIS 기간별시세는 요청 범위와 무관하게 **최근 100봉**까지만 반환합니다 — 장기 히스토리 분석에는 부족할 수 있습니다.

## 1. 일봉 데이터 로드 (KIS 기간별시세 — 계좌번호 불필요)

In [ ]:
import os
from datetime import datetime, timedelta

import dotenv
import pandas as pd
from pydantic import SecretStr

from cluefin_openapi.kis._auth import Auth as KisAuth
from cluefin_openapi.kis._http_client import HttpClient as KisClient

dotenv.load_dotenv(dotenv.find_dotenv(".env.test", usecwd=True))

env = os.getenv("KIS_ENV", "prod")
app_key = os.environ["KIS_APP_KEY"]
secret_key = os.environ["KIS_SECRET_KEY"]

auth = KisAuth(app_key=app_key, secret_key=SecretStr(secret_key), env=env)
token = auth.generate()  # 파일 캐시 우선 — 만료 전 재발급 없음
client = KisClient(token=token.get_token(), app_key=app_key, secret_key=SecretStr(secret_key), env=env)

STOCK_CODE = "005930"  # 삼성전자


def load_daily(stk_cd: str, days: int = 200) -> pd.DataFrame:
    """KIS 기간별시세(FHKST03010100)로 일봉 로드 — 최근 100봉 캡."""
    end = datetime.now()
    bars = client.domestic_basic_quote.get_stock_period_quote(
        fid_cond_mrkt_div_code="J",
        fid_input_iscd=stk_cd,
        fid_input_date_1=(end - timedelta(days=days)).strftime("%Y%m%d"),
        fid_input_date_2=end.strftime("%Y%m%d"),
        fid_period_div_code="D",
        fid_org_adj_prc="0",  # 수정주가
    ).body.output2
    rows = [
        {
            "date": pd.to_datetime(b.stck_bsop_date),
            "open": float(b.stck_oprc),
            "high": float(b.stck_hgpr),
            "low": float(b.stck_lwpr),
            "close": float(b.stck_clpr),
            "volume": float(b.acml_vol),
        }
        for b in bars
        if b.stck_bsop_date
    ]
    return pd.DataFrame(rows).set_index("date").sort_index()


df = load_daily(STOCK_CODE)
print(f"{len(df)} bars, {df.index.min():%Y-%m-%d} ~ {df.index.max():%Y-%m-%d}")
df.tail()

## 2. 추세 — 이동평균 (SMA·EMA)

In [ ]:
import cluefin_ta as ta
import plotext as plt

close = df["close"].to_numpy()
df["sma20"] = ta.SMA(close, timeperiod=20)
df["sma50"] = ta.SMA(close, timeperiod=50)
df["ema12"] = ta.EMA(close, timeperiod=12)

tail = df.tail(120)
plt.clear_figure()
plt.plot(tail["close"].tolist(), label="close")
plt.plot(tail["sma20"].tolist(), label="SMA20")
plt.plot(tail["sma50"].tolist(), label="SMA50")
plt.title(f"{STOCK_CODE} price / moving averages")
plt.plotsize(100, 25)
plt.show()

trend = "골든크로스(단기>장기) 상태" if tail["sma20"].iloc[-1] > tail["sma50"].iloc[-1] else "데드크로스(단기<장기) 상태"
print("해석:", trend)

## 3. 모멘텀 — RSI · MACD · 스토캐스틱

In [ ]:
df["rsi"] = ta.RSI(close, timeperiod=14)
macd, macd_signal, macd_hist = ta.MACD(close)
df["macd"], df["macd_signal"], df["macd_hist"] = macd, macd_signal, macd_hist
slowk, slowd = ta.STOCH(df["high"].to_numpy(), df["low"].to_numpy(), close)
df["stoch_k"], df["stoch_d"] = slowk, slowd

tail = df.tail(120)
plt.clear_figure()
plt.plot(tail["rsi"].tolist(), label="RSI(14)")
plt.hline(70)
plt.hline(30)
plt.title("RSI — 70 이상 과매수 / 30 이하 과매도")
plt.plotsize(100, 18)
plt.show()

last = df.iloc[-1]
rsi_zone = "과매수" if last.rsi >= 70 else "과매도" if last.rsi <= 30 else "중립"
macd_state = "시그널 상회(매수 우위)" if last.macd > last.macd_signal else "시그널 하회(매도 우위)"
print(f"RSI {last.rsi:.1f} → {rsi_zone} | MACD {last.macd:.1f} vs signal {last.macd_signal:.1f} → {macd_state}")
print(f"Stochastic %K {last.stoch_k:.1f} / %D {last.stoch_d:.1f}")

## 4. 변동성 — 볼린저 밴드 · ATR

In [ ]:
upper, middle, lower = ta.BBANDS(close, timeperiod=20)
df["bb_upper"], df["bb_middle"], df["bb_lower"] = upper, middle, lower
df["atr"] = ta.ATR(df["high"].to_numpy(), df["low"].to_numpy(), close, timeperiod=14)

tail = df.tail(120)
plt.clear_figure()
plt.plot(tail["close"].tolist(), label="close")
plt.plot(tail["bb_upper"].tolist(), label="BB upper")
plt.plot(tail["bb_lower"].tolist(), label="BB lower")
plt.title("Bollinger Bands (20, 2)")
plt.plotsize(100, 22)
plt.show()

last = df.iloc[-1]
band_pos = (last.close - last.bb_lower) / (last.bb_upper - last.bb_lower) * 100
print(f"밴드 내 위치 {band_pos:.0f}% (0%=하단, 100%=상단) | ATR(14) {last.atr:,.0f}원")

## 5. 리스크 지표 — MDD · CAGR · 변동성 · 샤프

In [ ]:
returns = df["close"].pct_change().dropna().to_numpy()
print(f"MDD        : {ta.MDD(returns):>8.2%}")
print(f"CAGR       : {ta.CAGR(returns):>8.2%}")
print(f"연변동성    : {ta.VOLATILITY(returns):>8.2%}")
print(f"Sharpe     : {ta.SHARPE(returns):>8.2f}")

## 6. 캔들 패턴 감지

In [ ]:
o, h, l, c = (df[k].to_numpy() for k in ("open", "high", "low", "close"))
patterns = {
    "도지": ta.CDLDOJI(o, h, l, c),
    "해머": ta.CDLHAMMER(o, h, l, c),
    "장악형": ta.CDLENGULFING(o, h, l, c),
    "샛별형": ta.CDLMORNINGSTAR(o, h, l, c),
}
for name, signal in patterns.items():
    hits = df.index[signal != 0]
    recent = ", ".join(d.strftime("%m-%d") for d in hits[-3:]) or "없음"
    print(f"{name:<4s}: 최근 발생일 {recent}")

## 7. ML 피처 엔지니어링 예시

cluefin-desk 의 ML 예측 탭(구 cluefin-cli `ta --ml-predict`)이 쓰는 것과 같은 계열의 피처 —
지표 + 랙(lag) + 롤링 통계 + 레짐 감지 — 를 직접 만들어 봅니다.

In [ ]:
feat = df[["close", "volume", "rsi", "macd_hist", "atr"]].copy()

# 수익률·랙 피처
feat["return_1d"] = feat["close"].pct_change()
for lag in (1, 3, 5):
    feat[f"return_lag{lag}"] = feat["return_1d"].shift(lag)

# 롤링 통계
feat["volatility_20d"] = feat["return_1d"].rolling(20).std()
feat["volume_ratio"] = feat["volume"] / feat["volume"].rolling(20).mean()

# 레짐 감지 (cluefin-ta): 이동평균 기반 상승/하락 국면
feat["regime_ma"] = ta.REGIME_MA(df["close"].to_numpy())

# 타깃: 익일 상승 여부 (desk ML 예측 탭과 동일한 정의)
feat["target_next_up"] = (feat["close"].shift(-1) > feat["close"]).astype(int)

feat = feat.dropna()
print(feat.shape)
feat.tail()